# Confirm the chosen encoder layer on the test split, once

Notebook 09 chose an encoder layer for the nearest-neighbour reference on the **synthetic validation set** (`results/improvements/layer_scan/selected_layer.json`). This notebook scores that layer on the official test split a single time, against the same reference on the final cached latents, with paired-bootstrap 95% intervals.

* **Reference row**: the notebook 03 PatchCore-style score on the cached final-layer latents. It must reproduce `results/evaluation/reference_comparison.csv`, otherwise this notebook is not measuring what notebook 03 measured.
* **Selected row**: the layer and feature treatment in the JSON; same memory bank construction (every nominal training token), same distance, same metrics.
* The test split played no part in choosing the layer. The notebook stops if the selection file says otherwise.

This is the kNN *reference*, an analysis tool with a memory bank. It shows how much more signal the chosen layer holds, not what a memory-free predictor achieves; that is a separate experiment (a predictor trained on the chosen layer).

In [1]:
import json
import os
import sys
import time
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image

try:
    display
except NameError:
    display = print

candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next((p for p in candidates if (p / 'configs').is_dir() and (p / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the repository or its notebooks directory.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

CATEGORIES = ('bottle', 'capsule', 'hazelnut')
ENCODER_CHECKPOINT = PROJECT_ROOT / 'checkpoints' / 'IN22K-vit.g.16-600e.pth.tar'
CACHE_ROOT = PROJECT_ROOT / 'data' / 'cache' / 'latents'
TEST_CACHE_ROOT = PROJECT_ROOT / 'data' / 'cache' / 'latents_test'
LAYER_ROOT = PROJECT_ROOT / 'data' / 'cache' / 'layer_features'
SELECTION = PROJECT_ROOT / 'results' / 'improvements' / 'layer_scan' / 'selected_layer.json'
OUTPUT_ROOT = Path(os.environ.get('LAYER_CONFIRM_OUTPUT', PROJECT_ROOT / 'results' / 'improvements' / 'layer_confirmation'))
OVERWRITE = False
OUTPUT_FILES = ('layer_vs_final.csv', 'bootstrap_deltas.csv', 'layer_confirmation_summary.json')
DATA_CONFIG = yaml.safe_load((PROJECT_ROOT / 'configs' / 'data' / 'mvtec.yaml').read_text(encoding='utf-8'))
VAL_FRACTION, SPLIT_SEED = float(DATA_CONFIG['nominal_split']['val_fraction']), int(DATA_CONFIG['nominal_split']['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_BOOTSTRAP, BOOTSTRAP_SEED, CHANNEL_STD_EPS, QUERY_CHUNK = 1000, 0, 1e-6, 12

if not OVERWRITE:
    existing = [OUTPUT_ROOT / n for n in OUTPUT_FILES if (OUTPUT_ROOT / n).exists()]
    if existing:
        raise FileExistsError('Refusing to overwrite: ' + ', '.join(map(str, existing)) + ' (set OVERWRITE = True)')
selection = json.loads(SELECTION.read_text(encoding='utf-8'))
if selection['test_used_for_selection'] is not False or selection['test_split_read'] is not False:
    raise ValueError('The selection file says the test split was involved.')
OPTION, TREATMENT = selection['selected']['option'], selection['selected']['treatment']
if OPTION == 'final (cached)':
    raise ValueError('The selected option is the final layer itself; there is nothing to confirm.')
LAYER = int(OPTION.split()[1])
K = int(json.loads((PROJECT_ROOT / 'results' / 'improvements' / 'synthetic_val' / 'selected_settings.json').read_text(encoding='utf-8'))['selected_settings']['k'])
print(f'selected on synthetic validation data: layer {LAYER}, treatment {TREATMENT!r}; top-k = {K}')

selected on synthetic validation data: layer 32, treatment 'layer norm'; top-k = 3


In [2]:
from ijepa_ad.caching.cache_schema import CacheExpectations, PreprocessingSpec
from ijepa_ad.caching.latent_cache import compute_file_sha256
from ijepa_ad.data.mvtec_dataset import validate_category_splits
from ijepa_ad.data.splits import split_nominal_train_val
from ijepa_ad.data.transforms import PreprocessConfig
from ijepa_ad.encoder.layer_features import load_layer_features
from ijepa_ad.evaluation.metrics import aupro, image_auroc, pixel_auroc
from ijepa_ad.evaluation.runner import EMBEDDING_DIM, ENCODER_NAME, NUM_TOKENS, PATCH_SIZE
from ijepa_ad.masking.score_maps import upsample_patch_scores
from ijepa_ad.predictor.cache_dataset import load_cached_latent_dataset

ENCODER_SHA256 = compute_file_sha256(ENCODER_CHECKPOINT)
SPEC = PreprocessingSpec.from_config(PreprocessConfig(image_size=int(DATA_CONFIG['preprocessing']['image_size'])))


def expectations(category, split, samples):
    return CacheExpectations(category=category, split=split, encoder_name=ENCODER_NAME, checkpoint_sha256=ENCODER_SHA256, patch_size=PATCH_SIZE,
                             embedding_dim=EMBEDDING_DIM, preprocessing=SPEC, sample_ids=tuple(s.image_id for s in samples), num_tokens=NUM_TOKENS)


def load_ground_truth(annotation):
    if annotation.mask_path is None:
        with Image.open(annotation.image_path) as image:
            width, height = image.size
        return np.zeros((height, width), dtype=bool)
    with Image.open(annotation.mask_path) as mask_image:
        return np.asarray(mask_image.convert('L')) > 0


DATA = {}
for category in CATEGORIES:
    root = next(r for r in (PROJECT_ROOT / DATA_CONFIG['root'], PROJECT_ROOT / 'dataset', PROJECT_ROOT / 'dataset' / category) if (r / category / 'train').is_dir())
    train_ds, test_ds = validate_category_splits(root, category)
    nominal = split_nominal_train_val(train_ds.samples, val_fraction=VAL_FRACTION, seed=SPLIT_SEED,
                                      min_train=int(DATA_CONFIG['nominal_split']['min_train']), min_val=int(DATA_CONFIG['nominal_split']['min_val']))
    train_final = load_cached_latent_dataset(CACHE_ROOT, expectations(category, 'train', nominal.train), samples=train_ds.samples).latents.float()
    test_cache = load_cached_latent_dataset(TEST_CACHE_ROOT, expectations(category, 'test', test_ds.samples), samples=test_ds.samples)
    test_ids = [s.image_id for s in test_ds.samples]
    spec = asdict(SPEC)
    layer_train = load_layer_features(LAYER_ROOT, layer=LAYER, category=category, split='train', sample_ids=[s.image_id for s in nominal.train],
                                      checkpoint_sha256=ENCODER_SHA256, preprocessing=spec).features.float()
    layer_test = load_layer_features(LAYER_ROOT, layer=LAYER, category=category, split='test', sample_ids=list(test_cache.sample_ids),
                                     checkpoint_sha256=ENCODER_SHA256, preprocessing=spec).features.float()
    DATA[category] = {
        'train_final': train_final, 'test_final': test_cache.latents.float(), 'train_layer': layer_train, 'test_layer': layer_test,
        'labels': np.asarray([int(s.annotation.label != 'good') for s in test_cache]),
        'masks': [load_ground_truth(s.annotation) for s in test_cache],
    }
    print(f'{category}: train {len(train_final)}, test {len(test_cache)}; layer {LAYER} features loaded')

bottle: train 188, test 83; layer 32 features loaded
capsule: train 197, test 132; layer 32 features loaded
hazelnut: train 352, test 110; layer 32 features loaded


## Scoring

Same as notebook 03 section 9b: bank = all nominal training tokens, score = L2 distance to the nearest bank token, image score = max or top-`k` mean of the 196 token scores, pixel metrics against the full-resolution ground truth (AUPRO with 200 thresholds).

In [3]:
def treat(treatment, train, test):
    if treatment == 'layer norm':
        return train, test
    flat = train.reshape(-1, train.shape[-1])
    mean, std = flat.mean(0), flat.std(0).clamp_min(CHANNEL_STD_EPS)
    return (train - mean) / std, (test - mean) / std


def knn_scores(train, test):
    bank = train.reshape(-1, train.shape[-1]).to(DEVICE)
    out = []
    with torch.inference_mode():
        for i in range(0, len(test), QUERY_CHUNK):
            chunk = test[i:i + QUERY_CHUNK].to(DEVICE).reshape(-1, test.shape[-1])
            out.append(torch.cdist(chunk, bank).min(dim=1).values.reshape(-1, test.shape[1]).cpu())
    return torch.cat(out)


def evaluate(category, scores):
    labels, masks = DATA[category]['labels'], DATA[category]['masks']
    maps = [upsample_patch_scores(p, output_size=m.shape)[0, 0].numpy().astype(np.float32) for p, m in zip(scores, masks)]
    return {'scores': {'max': scores.max(dim=1).values.numpy(), f'top{K}': scores.topk(K, dim=1).values.mean(dim=1).numpy()},
            'pixel_auroc': pixel_auroc(maps, masks), 'aupro': aupro(maps, masks, num_thresholds=200)}


RESULTS = {}
for category in CATEGORIES:
    started = time.perf_counter()
    d = DATA[category]
    RESULTS[(category, 'final (cached)')] = evaluate(category, knn_scores(d['train_final'], d['test_final']))
    train, test = treat(TREATMENT, d['train_layer'], d['test_layer'])
    RESULTS[(category, OPTION)] = evaluate(category, knn_scores(train, test))
    print(f'{category}: scored and evaluated in {time.perf_counter() - started:.0f}s')

rows = []
for category in CATEGORIES:
    labels = DATA[category]['labels']
    for name in ('final (cached)', OPTION):
        r = RESULTS[(category, name)]
        rows.append({'category': category, 'features': name if name == 'final (cached)' else f'{name} ({TREATMENT})',
                     'image_auroc_max': image_auroc(labels, r['scores']['max']), f'image_auroc_top{K}': image_auroc(labels, r['scores'][f'top{K}']),
                     'pixel_auroc': r['pixel_auroc'], 'aupro': r['aupro']})
metrics_df = pd.DataFrame(rows)
display(metrics_df.round(4))

saved = pd.read_csv(PROJECT_ROOT / 'results' / 'evaluation' / 'reference_comparison.csv')
saved = saved[saved.predictor == 'patchcore_style'].set_index('category')
check = metrics_df[metrics_df.features == 'final (cached)'].set_index('category')
REFERENCE_REPRODUCED = bool(all(
    abs(check.loc[c, 'pixel_auroc'] - saved.loc[c, 'pixel_auroc']) < 1e-4 and abs(check.loc[c, 'aupro'] - saved.loc[c, 'aupro']) < 1e-4
    and abs(check.loc[c, 'image_auroc_max'] - saved.loc[c, 'image_auroc']) < 1e-4 for c in CATEGORIES))
print('final-layer reference reproduces results/evaluation/reference_comparison.csv:', REFERENCE_REPRODUCED)
if not REFERENCE_REPRODUCED:
    raise AssertionError('The final-layer reference does not reproduce notebook 03.')

bottle: scored and evaluated in 40s
capsule: scored and evaluated in 85s
hazelnut: scored and evaluated in 58s
final-layer reference reproduces results/evaluation/reference_comparison.csv: True


,category,features,image_auroc_max,image_auroc_top3,pixel_auroc,aupro
0,bottle,final (cached),0.9992,1.0000,0.9727,0.8588
1,bottle,layer 32 (layer norm),1.0000,1.0000,0.9872,0.9371
2,capsule,final (cached),0.8045,0.8357,0.9155,0.5128
3,capsule,layer 32 (layer norm),0.9095,0.9095,0.9619,0.7705
4,hazelnut,final (cached),0.9875,0.9879,0.9791,0.8680
5,hazelnut,layer 32 (layer norm),0.9989,1.0000,0.9946,0.9408


## Paired bootstrap

Selected layer against the final-layer reference, on the same resampled test images (1,000 resamples, seed 0, good and anomalous resampled separately). `improves` means the whole interval is above 0.

In [4]:
def bootstrap_indices(labels):
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    pos, neg = np.flatnonzero(labels == 1), np.flatnonzero(labels == 0)
    return [np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]) for _ in range(N_BOOTSTRAP)]


def verdict(low, high):
    return 'improves' if low > 0 else 'worsens' if high < 0 else 'inconclusive (CI contains 0)'


boot_rows = []
for category in CATEGORIES:
    labels = DATA[category]['labels']
    indices = bootstrap_indices(labels)
    for key, label in (('max', 'max score'), (f'top{K}', f'top-{K} score')):
        new, ref = RESULTS[(category, OPTION)]['scores'][key], RESULTS[(category, 'final (cached)')]['scores'][key]
        deltas = np.asarray([image_auroc(labels[i], new[i]) - image_auroc(labels[i], ref[i]) for i in indices])
        low, high = np.percentile(deltas, [2.5, 97.5])
        boot_rows.append({'category': category, 'image_score': label, 'image_auroc': image_auroc(labels, new), 'reference_auroc': image_auroc(labels, ref),
                          'delta_image_auroc': image_auroc(labels, new) - image_auroc(labels, ref), 'ci95_low': low, 'ci95_high': high, 'verdict': verdict(low, high)})
boot_df = pd.DataFrame(boot_rows)
shown = boot_df.copy()
shown['CI95'] = shown.apply(lambda r: f'[{r.ci95_low:+.3f}, {r.ci95_high:+.3f}]', axis=1)
display(shown[['category', 'image_score', 'image_auroc', 'reference_auroc', 'delta_image_auroc', 'CI95', 'verdict']].round(4))

confirmation = PROJECT_ROOT / 'results' / 'improvements' / 'confirmation' / 'selected_vs_baseline.csv'
if confirmation.exists():
    print('\nfor context, our trained predictor with the notebook 06 settings (notebook 07):')
    display(pd.read_csv(confirmation).query("config == 'selected'").round(4))


for context, our trained predictor with the notebook 06 settings (notebook 07):


,category,image_score,image_auroc,reference_auroc,delta_image_auroc,CI95,verdict
0,bottle,max score,1.0000,0.9992,0.0008,"[+0.000, +0.005]",inconclusive (CI contains 0)
1,bottle,top-3 score,1.0000,1.0000,0.0000,"[+0.000, +0.000]",inconclusive (CI contains 0)
2,capsule,max score,0.9095,0.8045,0.1049,"[+0.022, +0.187]",improves
3,capsule,top-3 score,0.9095,0.8357,0.0738,"[+0.017, +0.135]",improves
4,hazelnut,max score,0.9989,0.9875,0.0114,"[-0.001, +0.036]",inconclusive (CI contains 0)
5,hazelnut,top-3 score,1.0000,0.9879,0.0121,"[+0.000, +0.039]",inconclusive (CI contains 0)


,category,config,image_auroc_max,image_auroc_top3,pixel_auroc,aupro,masks_per_image,scorer_ms_per_image
1,bottle,selected,0.9929,0.9976,0.9524,0.8309,16.1928,22.4659
3,capsule,selected,0.6223,0.6721,0.9011,0.5080,16.0000,23.0953
5,hazelnut,selected,0.8643,0.9393,0.9692,0.8118,16.1455,23.5091


In [5]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not OVERWRITE:
    existing = [OUTPUT_ROOT / n for n in OUTPUT_FILES if (OUTPUT_ROOT / n).exists()]
    if existing:
        raise FileExistsError('Refusing to overwrite: ' + ', '.join(map(str, existing)))
metrics_df.to_csv(OUTPUT_ROOT / 'layer_vs_final.csv', index=False)
boot_df.to_csv(OUTPUT_ROOT / 'bootstrap_deltas.csv', index=False)
(OUTPUT_ROOT / 'layer_confirmation_summary.json').write_text(json.dumps({
    'selected_on': selection['chosen_from'], 'selected': selection['selected'], 'test_used_for_selection': False,
    'test_split_scored': 'once, for the final-layer reference and the selected layer only',
    'final_layer_reference_reproduces_notebook_03': REFERENCE_REPRODUCED,
    'bootstrap': {'n': N_BOOTSTRAP, 'seed': BOOTSTRAP_SEED},
    'verdicts': boot_df.set_index(['category', 'image_score']).verdict.to_dict().__repr__(),
}, indent=2, sort_keys=True, default=str), encoding='utf-8')
print('saved to', OUTPUT_ROOT)

saved to C:\Users\aryan\OneDrive\Desktop\7 sem\I-JEPA-AD\results\improvements\layer_confirmation


## What you can and cannot say

**You can say** whether the layer chosen on synthetic validation data holds more nearest-neighbour signal than the final layer on the real test split, with intervals, and that the test split did not influence the choice.

**You cannot say**

* That a predictor trained on this layer will profit: the kNN reference uses a memory bank of all nominal training tokens, and the predictor has to extract the signal without it.
* That this layer is best among all possible layers: nine layers were scanned and ties were broken toward the later layer.
* That an interval containing 0 shows a gain or a loss.